In [1]:
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "simulation_modeling" / "v01_model.py").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")

repo_root = find_repo_root(Path.cwd())
sys.path.insert(0, str(repo_root))

import numpy as np
from simulation_modeling import v01_model
from simulation_modeling.candidate_sweep_utils import (
    CandidateResult,
    CandidateSpec,
    ModelParams,
    RejectLimits,
    ScoreWeights,
    SweepSpec,
    evaluate_candidate_over_sweep,
    export_candidate_results,
    generate_two_guide_candidates,
    make_coordinated_flexion_sweep,
    make_tendon_input,
    run_experiment,
    save_plotly_figure_bundle,
    scale_finger_geometry,
    summarize_candidate_output,
)


In [2]:
# I want to run this sweep let's say 3 times with 3 different finger lengths all scaled the same lets say and then also with different stiffnesses per completed sweep
# so like an 3 * 3 or whatever


'''
Proximal Interphalangeal Joint Stiffness: Measurement and Analysis
https://www.sciencedirect.com/science/article/abs/pii/S0363502304007671

average overall stiffness = 0.05 N-cm/degree

No robust tapping-specific average passive MCP/PIP/DIP stiffness set was identified as a direct literature constant. 
Instead, the model uses literature-informed nominal stiffness ranges and treats passive joint stiffness as a parameter to be swept and later calibrated. 
This is consistent with prior finger biomechanics literature, 
where passive resistance is often represented by nonlinear torque-angle laws and only later linearized for reduced-order simulation.



'''
 
IDX_L_PROX = 41.0
IDX_L_MIDDLE = 26
IDX_L_DIST = 20

IDX_D_PROX = 0.33 * IDX_L_PROX
IDX_D_MIDDLE  = 0.78 * IDX_L_MIDDLE
IDX_D_DIST = 0.68 * IDX_L_DIST

def scale_finger_geometry(
    geom: v01_model.FingerGeometry,
    scale: float,
) -> v01_model.FingerGeometry:
    return v01_model.FingerGeometry(
        geom.l1 * scale,
        geom.l2 * scale,
        geom.l3 * scale,
        geom.d1 * scale,
        geom.d2 * scale,
        geom.d3 * scale,
    )

BASE_FINGER_GEOM = v01_model.FingerGeometry(
    IDX_L_PROX,
    IDX_L_MIDDLE,
    IDX_L_DIST,
    IDX_D_PROX,
    IDX_D_MIDDLE,
    IDX_D_DIST,
)


FINGER_GEOMS = {
    "small": scale_finger_geometry(BASE_FINGER_GEOM, 0.9),
    "nominal": BASE_FINGER_GEOM,
    "large": scale_finger_geometry(BASE_FINGER_GEOM, 1.1),
}

JOINT_REST = (0.0, 0.0, 0.0)

NOMINAL_K_PASSIVE = np.array([0.02, 0.0286, 0.015], dtype=float) #  N·m/rad
MID_K_PASSIVE = 1.15 * NOMINAL_K_PASSIVE
HIGH_K_PASSIVE = 1.3 * NOMINAL_K_PASSIVE

JOINT_STIFFNESSES = {
    "nominal": tuple(NOMINAL_K_PASSIVE),
    "mid": tuple(MID_K_PASSIVE),
    "high": tuple(HIGH_K_PASSIVE),
}

MODEL_CASES = []

for size_name, geom in FINGER_GEOMS.items():
    for stiff_name, stiffness in JOINT_STIFFNESSES.items():
        MODEL_CASES.append(
            ModelParams(
                name=f"{size_name}_{stiff_name}",
                size_name=f"{size_name}",
                stiffness_name=f"{stiff_name}",
                geom=geom,
                joint_rest=JOINT_REST,
                stiffness=stiffness,
            )
        )

In [3]:
SWEEP_SPECS = [
    SweepSpec(
        "flexion_sweep",
        q_sweep=make_coordinated_flexion_sweep(),
    )
]


In [4]:
TWO_GUIDE_CANDIDATES = generate_two_guide_candidates(
    prox_long_values=[0.25, 0.45, 0.65],
    mid_long_values=[0.25, 0.45, 0.65],
    offset_values=[-3.0, -5.0],
    distal_anchor_long_values=[0.80],
    distal_anchor_offset_values=[-4.0],
    tendon_entry_values=[(-8.0, -4.0)],
)


In [5]:
test_model = MODEL_CASES[0]
test_candidate = TWO_GUIDE_CANDIDATES[0]
test_q = SWEEP_SPECS[0].q_sweep[10]

test_tendon = make_tendon_input(test_candidate, test_model)
test_fk = v01_model.forward_kinematics(test_model.geom, test_q)


In [6]:
REJECT_LIMITS = RejectLimits(
    max_tension=10.0,
    max_pull=100.0,
    max_tau_error=1.0,
    max_branch_imbalance=20.0,
    min_moment_arm_norm=1e-6,
)

SCORE_WEIGHTS = ScoreWeights(
    w_tension=1.0,
    w_tau_error=1.0,
    w_pull=1.0,
    w_balance=1.0,
    w_smoothness=0.1,
)

sample_output = evaluate_candidate_over_sweep(test_candidate, test_model, SWEEP_SPECS[0])
sample_result = summarize_candidate_output(sample_output, REJECT_LIMITS, SCORE_WEIGHTS)
sample_result


CandidateResult(model_case_name='small_nominal', candidate_name='twoG_b-p_m_l-0.25_0.25_o--3.00_-3.00_al-0.80_ao--4.00_ex--8.00_ey--4.00', sweep_name='flexion_sweep', peak_tension=2.3069015479082586, mean_tension=1.4490524231090494, peak_pull=14.371467058646871, peak_branch_imbalance=0.0, tau_error_rms=0.009895618664435269, tau_error_peak=0.017909119092661286, tau_error_rel_rms=0.8431247521205153, tau_error_rel_peak=0.9240556188237101, score=1.2370094669351999, rejected=False, reject_reasons=[])

In [7]:
experiment = run_experiment(
    model_cases=MODEL_CASES,
    candidates=TWO_GUIDE_CANDIDATES,
    sweep_specs=SWEEP_SPECS,
    reject_limits=REJECT_LIMITS,
    score_weights=SCORE_WEIGHTS,
)

raw_path, summary_path, raw_history_path, summary_history_path = export_candidate_results(
    experiment.outputs,
    experiment.results,
    run_label="two_guide_first_pass",
)

experiment.results[:5]


[CandidateResult(model_case_name='large_nominal', candidate_name='twoG_b-p_m_l-0.25_0.65_o--5.00_-5.00_al-0.80_ao--4.00_ex--8.00_ey--4.00', sweep_name='flexion_sweep', peak_tension=1.770120701442527, mean_tension=1.1049709091505089, peak_pull=21.392824661115867, peak_branch_imbalance=0.0, tau_error_rms=0.0029834289050719737, tau_error_peak=0.004437911891509675, tau_error_rel_rms=0.43199444173954044, tau_error_rel_peak=0.6447790082272614, score=0.8542392093805087, rejected=False, reject_reasons=[]),
 CandidateResult(model_case_name='large_nominal', candidate_name='twoG_b-p_m_l-0.25_0.65_o--3.00_-3.00_al-0.80_ao--4.00_ex--8.00_ey--4.00', sweep_name='flexion_sweep', peak_tension=1.9861414696112718, mean_tension=1.3085442179848195, peak_pull=17.82221248786763, peak_branch_imbalance=0.0, tau_error_rms=0.0028759588878505015, tau_error_peak=0.003966403439762, tau_error_rel_rms=0.4470307105091187, tau_error_rel_peak=0.7413239237686308, score=0.8561780368841712, rejected=False, reject_reasons=[

In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Keep plotting inline in the notebook; avoid external browser renderers.
pio.renderers.default = "notebook_connected"

FIGURE_RUN_LABEL = "2026_04_30_two_guide_screening"

# Build a compact ranking table for tonight's review.
summary_df = pd.DataFrame([
    {
        "model_case": r.model_case_name,
        "candidate": r.candidate_name,
        "sweep": r.sweep_name,
        "peak_tension": r.peak_tension,
        "mean_tension": r.mean_tension,
        "peak_pull_mm": r.peak_pull,
        "tau_error_rms": r.tau_error_rms,
        "tau_error_peak": r.tau_error_peak,
        "tau_error_rel_rms": r.tau_error_rel_rms,
        "tau_error_rel_peak": r.tau_error_rel_peak,
        "score": r.score,
        "rejected": r.rejected,
        "reject_reasons": "|".join(r.reject_reasons),
    }
    for r in experiment.results
])

top_df = summary_df.sort_values(["rejected", "score", "peak_tension"]).head(10).reset_index(drop=True)
top_df

# Inspect the best non-rejected candidate in detail.
best_result = next(r for r in experiment.results if not r.rejected)
best_output = next(
    o for o in experiment.outputs
    if o.model_case_name == best_result.model_case_name
    and o.candidate_name == best_result.candidate_name
    and o.sweep_name == best_result.sweep_name
)

posture_idx = np.arange(len(best_output.q_sweep))
tau_err_norm = np.linalg.norm(best_output.tau_error, axis=1)
tau_err_rel_norm = np.linalg.norm(best_output.tau_error_rel, axis=1)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Required Tension Over Sweep",
        "Torque Error Over Sweep",
        "Total Moment-Arm Components (mm/rad)",
        "Fingertip Trajectory",
    ),
)

fig.add_trace(
    go.Scatter(x=posture_idx, y=best_output.tension_req, mode="lines", name="T_req [N]"),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(x=posture_idx, y=tau_err_norm, mode="lines", name="||tau_error|| [N·m]"),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(x=posture_idx, y=tau_err_rel_norm, mode="lines", name="||tau_error_rel|| [-]"),
    row=1, col=2,
)

fig.add_trace(
    go.Scatter(x=posture_idx, y=best_output.r_total[:, 0], mode="lines", name="R_MCP"),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=posture_idx, y=best_output.r_total[:, 1], mode="lines", name="R_PIP"),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(x=posture_idx, y=best_output.r_total[:, 2], mode="lines", name="R_DIP"),
    row=2, col=1,
)

fig.add_trace(
    go.Scatter(
        x=best_output.tip_traj[:, 0],
        y=best_output.tip_traj[:, 1],
        mode="lines+markers",
        name="tip path",
        marker=dict(size=4),
    ),
    row=2, col=2,
)

fig.update_xaxes(title_text="posture index", row=1, col=1)
fig.update_xaxes(title_text="posture index", row=1, col=2)
fig.update_xaxes(title_text="posture index", row=2, col=1)
fig.update_xaxes(title_text="x [mm]", row=2, col=2)
fig.update_yaxes(title_text="T_req [N]", row=1, col=1)
fig.update_yaxes(title_text="error", row=1, col=2)
fig.update_yaxes(title_text="R_total [mm/rad]", row=2, col=1)
fig.update_yaxes(title_text="y [mm]", row=2, col=2, scaleanchor="x", scaleratio=1)
fig.update_layout(height=850, width=1200, title_text=f"Best Candidate Analysis: {best_result.candidate_name} | {best_result.model_case_name}")
coarse_html_path, coarse_png_path = save_plotly_figure_bundle(
    fig,
    FIGURE_RUN_LABEL,
    "coarse_best_candidate_analysis",
)
fig.show()

coarse_html_path, coarse_png_path, best_result


CandidateResult(model_case_name='large_nominal', candidate_name='twoG_b-p_m_l-0.25_0.65_o--5.00_-5.00_al-0.80_ao--4.00_ex--8.00_ey--4.00', sweep_name='flexion_sweep', peak_tension=1.770120701442527, mean_tension=1.1049709091505089, peak_pull=21.392824661115867, peak_branch_imbalance=0.0, tau_error_rms=0.0029834289050719737, tau_error_peak=0.004437911891509675, tau_error_rel_rms=0.43199444173954044, tau_error_rel_peak=0.6447790082272614, score=0.8542392093805087, rejected=False, reject_reasons=[])

In [9]:
# Refined local pass around the current winner region.
REFINED_TWO_GUIDE_CANDIDATES = generate_two_guide_candidates(
    prox_long_values=[0.20, 0.25, 0.30],
    mid_long_values=[0.55, 0.65, 0.75],
    offset_values=[-4.0, -5.0, -6.0],
    distal_anchor_long_values=[0.80],
    distal_anchor_offset_values=[-4.0],
    tendon_entry_values=[(-8.0, -4.0)],
)

refined_experiment = run_experiment(
    model_cases=MODEL_CASES,
    candidates=REFINED_TWO_GUIDE_CANDIDATES,
    sweep_specs=SWEEP_SPECS,
    reject_limits=REJECT_LIMITS,
    score_weights=SCORE_WEIGHTS,
)

refined_summary_df = pd.DataFrame([
    {
        "model_case": r.model_case_name,
        "candidate": r.candidate_name,
        "sweep": r.sweep_name,
        "peak_tension": r.peak_tension,
        "mean_tension": r.mean_tension,
        "peak_pull_mm": r.peak_pull,
        "tau_error_rms": r.tau_error_rms,
        "tau_error_peak": r.tau_error_peak,
        "tau_error_rel_rms": r.tau_error_rel_rms,
        "tau_error_rel_peak": r.tau_error_rel_peak,
        "score": r.score,
        "rejected": r.rejected,
        "reject_reasons": "|".join(r.reject_reasons),
    }
    for r in refined_experiment.results
])

refined_top_df = refined_summary_df.sort_values(["rejected", "score", "peak_tension"]).head(10).reset_index(drop=True)
refined_top_df

refined_best_result = next(r for r in refined_experiment.results if not r.rejected)
refined_best_output = next(
    o for o in refined_experiment.outputs
    if o.model_case_name == refined_best_result.model_case_name
    and o.candidate_name == refined_best_result.candidate_name
    and o.sweep_name == refined_best_result.sweep_name
)

refined_posture_idx = np.arange(len(refined_best_output.q_sweep))
refined_tau_err_norm = np.linalg.norm(refined_best_output.tau_error, axis=1)
refined_tau_err_rel_norm = np.linalg.norm(refined_best_output.tau_error_rel, axis=1)

refined_fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Refined Required Tension Over Sweep",
        "Refined Torque Error Over Sweep",
        "Refined Total Moment-Arm Components (mm/rad)",
        "Refined Fingertip Trajectory",
    ),
)

refined_fig.add_trace(
    go.Scatter(x=refined_posture_idx, y=refined_best_output.tension_req, mode="lines", name="T_req [N]"),
    row=1, col=1,
)
refined_fig.add_trace(
    go.Scatter(x=refined_posture_idx, y=refined_tau_err_norm, mode="lines", name="||tau_error|| [N·m]"),
    row=1, col=2,
)
refined_fig.add_trace(
    go.Scatter(x=refined_posture_idx, y=refined_tau_err_rel_norm, mode="lines", name="||tau_error_rel|| [-]"),
    row=1, col=2,
)
refined_fig.add_trace(
    go.Scatter(x=refined_posture_idx, y=refined_best_output.r_total[:, 0], mode="lines", name="R_MCP"),
    row=2, col=1,
)
refined_fig.add_trace(
    go.Scatter(x=refined_posture_idx, y=refined_best_output.r_total[:, 1], mode="lines", name="R_PIP"),
    row=2, col=1,
)
refined_fig.add_trace(
    go.Scatter(x=refined_posture_idx, y=refined_best_output.r_total[:, 2], mode="lines", name="R_DIP"),
    row=2, col=1,
)
refined_fig.add_trace(
    go.Scatter(
        x=refined_best_output.tip_traj[:, 0],
        y=refined_best_output.tip_traj[:, 1],
        mode="lines+markers",
        name="tip path",
        marker=dict(size=4),
    ),
    row=2, col=2,
)
refined_fig.update_xaxes(title_text="posture index", row=1, col=1)
refined_fig.update_xaxes(title_text="posture index", row=1, col=2)
refined_fig.update_xaxes(title_text="posture index", row=2, col=1)
refined_fig.update_xaxes(title_text="x [mm]", row=2, col=2)
refined_fig.update_yaxes(title_text="T_req [N]", row=1, col=1)
refined_fig.update_yaxes(title_text="error", row=1, col=2)
refined_fig.update_yaxes(title_text="R_total [mm/rad]", row=2, col=1)
refined_fig.update_yaxes(title_text="y [mm]", row=2, col=2, scaleanchor="x", scaleratio=1)
refined_fig.update_layout(height=850, width=1200, title_text=f"Refined Best Candidate Analysis: {refined_best_result.candidate_name} | {refined_best_result.model_case_name}")
refined_html_path, refined_png_path = save_plotly_figure_bundle(
    refined_fig,
    FIGURE_RUN_LABEL,
    "refined_best_candidate_analysis",
)
refined_fig.show()

refined_html_path, refined_png_path, refined_best_result


CandidateResult(model_case_name='large_nominal', candidate_name='twoG_b-p_m_l-0.20_0.75_o--4.00_-4.00_al-0.80_ao--4.00_ex--8.00_ey--4.00', sweep_name='flexion_sweep', peak_tension=1.8190647399990312, mean_tension=1.1794864501023297, peak_pull=19.81783102340185, peak_branch_imbalance=0.0, tau_error_rms=0.0016399736919036785, tau_error_peak=0.0021330854045729765, tau_error_rel_rms=0.33916698039673515, tau_error_rel_peak=0.6791365716048219, score=0.7535069231537143, rejected=False, reject_reasons=[])